# ML-09 — Validation and Research Claim Audit

**Capstone lane:** Refresh / Content Opportunity Scoring  
**Decision window:** March 2026  
**Outcome window:** April 2026  
**Model:** Logistic Regression (Week-5 / ML-08)  
**Primary metric:** Precision@K (K = 10, 50, 100, 500)

This notebook keeps the established Week-5 lane, data contract, March→April outcome definition, and Logistic Regression model family. It audits the original random row split against a harder client-grouped validation design. The purpose is not to prove production performance; it is to measure how sensitive the Week-5 result is to validation design and to document leakage/failure risks.

> **Public-safe rule:** use observed, measured, directional, and decision-support language. Do not publish client names, URLs, queries, or row-level identifiers.

## 1. Two paper findings + my methodology questions

### Finding 1 — CTR changes sharply by search-position tier

The research material reports an observed CTR gradient across position tiers: CTR is substantially higher for pages in stronger search positions and falls as pages move deeper in results.

**My methodology question:** How was this comparison constructed — especially the inclusion threshold, treatment of pages with zero impressions, and whether the result is based on simple page-level means or an impression-weighted measure? I would want the same population and weighting rule applied consistently before interpreting the size of the difference.

**Why this is constructive:** Making the population and weighting explicit would help a reader distinguish a descriptive association from a broader claim about search behavior.

### Finding 2 — search volume and observed impressions are only weakly related

The research material reports an approximately zero correlation between keyword search volume and observed page impressions in the analyzed slice.

**My methodology question:** What exact rows were included in the correlation calculation, how were missing/zero values handled, and was the relationship checked for influential outliers or non-linear patterns? A near-zero Pearson correlation can be informative, but it does not by itself establish that search volume has no relationship with traffic under other conditions.

**Why this is constructive:** Reporting the population, missing-value treatment, and correlation choice would make the finding easier to reproduce and would prevent a descriptive statistic from being read as a causal conclusion.

**Scope note:** These are respectful methodology questions, not claims that the research finding is wrong. This section applies the same scrutiny we are applying to the Week-5 model.

## 2. My model under an honest split (before/after)

**Before:** reproduce the Week-5 stratified 80/20 row split. Because `client_hash_id` repeats across rows, the same client can occur in both train and test.

**After:** use `StratifiedGroupKFold` grouped by `client_hash_id`. Each validation row receives an out-of-fold prediction from a model trained without that row's client. This is a stronger test of generalization to unseen clients.

April is used only to construct the future-decline outcome. Predictive features are March-only. Client/content IDs are audit/grouping fields, never model features.

In [2]:
%pip -q install duckdb pandas numpy scikit-learn

import os
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is missing. Add your Read token to Colab Secrets as HF_TOKEN.')

con = duckdb.connect()
con.execute('INSTALL httpfs;')
con.execute('LOAD httpfs;')
con.execute('CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)', [HF_TOKEN])

MARCH_REL = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
APRIL_REL = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
RANDOM_STATE = 42
FEATURES = ['march_impressions','march_clicks','march_ctr_pct','march_avg_position','march_impression_days']
TARGET = 'future_decline_label'
GROUP = 'client_hash_id'
KS = [10, 50, 100, 500]

In [3]:
march_sql = f'''
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS march_impressions,
       SUM(gsc_clicks) AS march_clicks,
       CASE WHEN SUM(gsc_impressions) > 0 THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions) ELSE NULL END AS march_ctr_pct,
       CASE WHEN SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END) > 0
            THEN SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0 THEN gsc_impressions * gsc_avg_position ELSE 0 END)
                 / SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END)
            ELSE NULL END AS march_avg_position,
       COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS march_impression_days
FROM read_parquet('{MARCH_REL}')
WHERE month = '2026-03' AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
'''

april_sql = f'''
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS april_impressions
FROM read_parquet('{APRIL_REL}')
WHERE month = '2026-04' AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
'''

march = con.execute(march_sql).df()
april = con.execute(april_sql).df()
key_cols = ['client_hash_id','content_hash_id']
assert not march.duplicated(key_cols).any()
assert not april.duplicated(key_cols).any()
df = march.merge(april, on=key_cols, how='inner', validate='one_to_one')
df[TARGET] = ((df['march_impressions'] > 0) & (df['april_impressions'] < 0.80 * df['march_impressions'])).astype(int)

print(f'Rows: {len(df):,}')
print(f'Clients: {df[GROUP].nunique():,}')
print(f'Observed decline rate: {df[TARGET].mean():.4f}')
display(df[FEATURES + [TARGET]].describe().T.round(4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 158,549
Clients: 46
Observed decline rate: 0.4782


,count,mean,std,min,25%,50%,75%,max
march_impressions,158549.0,1768.0347,5706.7568,1.0000,40.00,246.0000,1264.0000,617124.0
march_clicks,158549.0,5.1767,28.1657,0.0000,0.00,0.0000,2.0000,5668.0
march_ctr_pct,158549.0,0.3719,2.5448,0.0000,0.00,0.0000,0.2514,100.0
march_avg_position,157790.0,16.9097,18.1783,0.0196,5.25,8.8622,22.1564,309.0
march_impression_days,158549.0,22.3052,10.4806,1.0000,14.00,29.0000,31.0000,31.0
future_decline_label,158549.0,0.4782,0.4995,0.0000,0.00,0.0000,1.0000,1.0


In [4]:
def precision_at_k(frame, score_col, k):
    ranked = frame.sort_values([score_col,'march_impressions','march_clicks'], ascending=[False,False,False])
    return float(ranked.head(min(k, len(ranked)))[TARGET].mean())

def make_model():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])

# BEFORE — Week-5 random row split
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.20, random_state=RANDOM_STATE, stratify=df[TARGET])
before_train, before_test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
before_model = make_model().fit(before_train[FEATURES], before_train[TARGET])
before_test['model_probability'] = before_model.predict_proba(before_test[FEATURES])[:,1]

# AFTER — client-grouped, stratified out-of-fold predictions
after = df.copy()
after['model_probability'] = np.nan
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr, va) in enumerate(cv.split(df[FEATURES], df[TARGET], groups=df[GROUP]), start=1):
    m = make_model().fit(df.iloc[tr][FEATURES], df.iloc[tr][TARGET])
    after.iloc[va, after.columns.get_loc('model_probability')] = m.predict_proba(df.iloc[va][FEATURES])[:,1]
    overlap = set(df.iloc[tr][GROUP]) & set(df.iloc[va][GROUP])
    assert not overlap, f'Client leakage in fold {fold}'

rows = []
for k in KS:
    rows.append({'evaluation':'Before — random row split','k':k,'precision_at_k':precision_at_k(before_test,'model_probability',k)})
    rows.append({'evaluation':'After — client-grouped OOF','k':k,'precision_at_k':precision_at_k(after,'model_probability',k)})
comparison = pd.DataFrame(rows)
comparison['precision_pct'] = (100 * comparison['precision_at_k']).round(2)
display(comparison)

secondary = pd.DataFrame([
    {'evaluation':'Before — random row split','ROC_AUC':roc_auc_score(before_test[TARGET], before_test['model_probability']),'Average_Precision':average_precision_score(before_test[TARGET], before_test['model_probability'])},
    {'evaluation':'After — client-grouped OOF','ROC_AUC':roc_auc_score(after[TARGET], after['model_probability']),'Average_Precision':average_precision_score(after[TARGET], after['model_probability'])},
])
display(secondary.round(4))
print('Clients overlapping between BEFORE train/test:', len(set(before_train[GROUP]) & set(before_test[GROUP])))

,evaluation,k,precision_at_k,precision_pct
0,Before — random row split,10,0.500,50.0
1,After — client-grouped OOF,10,0.600,60.0
2,Before — random row split,50,0.640,64.0
3,After — client-grouped OOF,50,0.640,64.0
4,Before — random row split,100,0.670,67.0
5,After — client-grouped OOF,100,0.600,60.0
6,Before — random row split,500,0.700,70.0
7,After — client-grouped OOF,500,0.566,56.6


,evaluation,ROC_AUC,Average_Precision
0,Before — random row split,0.6462,0.6059
1,After — client-grouped OOF,0.5900,0.5382


Clients overlapping between BEFORE train/test: 42


### Before/after interpretation

The key question is whether the Week-5 estimate changes when the same client is prevented from appearing in both training and validation. A lower grouped result is evidence that the random row split may have benefited from client overlap. A similar result would indicate more stability across unseen clients. Neither result establishes production performance or causal impact.

## 3. Leakage audit

The audit checks explicit future/label-derived fields, decision-time legality, identifier use, and the construction of the evaluation population.

In [5]:
leakage_checks = []
future_or_label = {'april_impressions','future_decline_label','trend_direction','trend_pct','label','target'}
bad_features = sorted(set(FEATURES) & future_or_label)
leakage_checks.append({'check':'Future/label-derived columns absent from features','status':'PASS' if not bad_features else 'FAIL','detail':str(bad_features)})

id_features = sorted(set(FEATURES) & {'client_hash_id','content_hash_id'})
leakage_checks.append({'check':'Client/content IDs excluded from predictive features','status':'PASS' if not id_features else 'FAIL','detail':str(id_features)})

march_only = all(c.startswith('march_') for c in FEATURES)
leakage_checks.append({'check':'Predictive features are March-only','status':'PASS' if march_only else 'FAIL','detail':str(FEATURES)})

label_uses_april = 'april_impressions' in df.columns and df[TARGET].notna().all()
leakage_checks.append({'check':'April appears only in the constructed outcome, not X','status':'PASS' if label_uses_april and 'april_impressions' not in FEATURES else 'FAIL','detail':'April impressions are used only to construct the future-decline label'})

key_unique = not df.duplicated(['client_hash_id','content_hash_id']).any()
leakage_checks.append({'check':'One row per client/content evaluation key','status':'PASS' if key_unique else 'FAIL','detail':'one-to-one March/April merge asserted'})

group_count = df[GROUP].nunique()
leakage_checks.append({'check':'Grouped validation has multiple clients','status':'PASS' if group_count > 1 else 'FAIL','detail':f'{group_count:,} clients'})

leakage_audit = pd.DataFrame(leakage_checks)
display(leakage_audit)
assert (leakage_audit['status'] == 'PASS').all(), 'Leakage audit failed; inspect the table before submission.'

,check,status,detail
0,Future/label-derived columns absent from features,PASS,[]
1,Client/content IDs excluded from predictive fe...,PASS,[]
2,Predictive features are March-only,PASS,"['march_impressions', 'march_clicks', 'march_c..."
3,"April appears only in the constructed outcome,...",PASS,April impressions are used only to construct t...
4,One row per client/content evaluation key,PASS,one-to-one March/April merge asserted
5,Grouped validation has multiple clients,PASS,46 clients


### Leakage conclusion

This notebook controls the main leakage routes identified for this lane: predictive features are March-only, the April window is used only for the outcome, identifiers are not model features, the March/April merge is one-to-one at the evaluation key, and grouped validation prevents client overlap between train and validation. This does not prove that every future feature addition will be leakage-free; the same decision-time audit must be repeated for future changes.

## 4. Real failure examples

The examples below are anonymized and show high-confidence mistakes under the honest grouped evaluation. They are evidence of model uncertainty, not explanations of why an individual page changed.

In [6]:
fp = after[(after['model_probability'] >= 0.5) & (after[TARGET] == 0)].sort_values('model_probability', ascending=False).head(5)
fn = after[(after['model_probability'] < 0.5) & (after[TARGET] == 1)].sort_values('model_probability', ascending=True).head(5)
show_cols = FEATURES + ['model_probability', TARGET]
print('False positives — high score, no observed decline:')
display(fp[show_cols].round(6))
print('False negatives — low score, observed decline:')
display(fn[show_cols].round(6))
print('Failure counts:', {'false_positives': int(((after['model_probability'] >= 0.5) & (after[TARGET] == 0)).sum()), 'false_negatives': int(((after['model_probability'] < 0.5) & (after[TARGET] == 1)).sum())})

False positives — high score, no observed decline:


,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days,model_probability,future_decline_label
273,134984.0,1.0,0.000741,2.693038,31,0.950030,0
112431,71513.0,3.0,0.004195,6.983444,31,0.855118,0
58731,55937.0,15.0,0.026816,21.659134,31,0.770636,0
27791,42185.0,4.0,0.009482,9.003058,30,0.764882,0
2346,28950.0,0.0,0.000000,9.563005,31,0.747423,0


False negatives — low score, observed decline:


,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days,model_probability,future_decline_label
64253,130338.0,1526.0,1.170802,3.176434,31,0.000000,1
35010,245276.0,1480.0,0.603402,2.757730,29,0.000000,1
69072,16986.0,1014.0,5.969622,6.265984,31,0.000000,1
58746,83012.0,827.0,0.996242,3.826350,31,0.000000,1
39039,82898.0,570.0,0.687592,3.725325,31,0.000004,1


Failure counts: {'false_positives': 29707, 'false_negatives': 37833}


### Failure interpretation

False positives show March signals that looked concerning but did not cross the measured April decline threshold. False negatives show cases where March signals looked comparatively healthy but the measured outcome still declined. These examples demonstrate uncertainty and boundary cases; they do not establish causal explanations.

## 5. Claim rewrite

### Original Week-5-style claim
> The logistic regression model improves ranking quality and can predict which pages will decline.

### Safer claim
On the tested March→April population, the model produced a measured ranking signal under both validation designs, but the client-grouped evaluation is the more conservative estimate for unseen-client generalization. The result should be described as **decision-support for prioritization**, not as a guarantee that an individual page will decline or that a refresh will improve performance.

The next code cell inserts the actual measured Precision@50 values after execution so the narrative is tied to the observed run rather than a hard-coded claim.

In [7]:
before50 = float(comparison.loc[(comparison['evaluation'].str.startswith('Before')) & (comparison['k'] == 50), 'precision_pct'].iloc[0])
after50 = float(comparison.loc[(comparison['evaluation'].str.startswith('After')) & (comparison['k'] == 50), 'precision_pct'].iloc[0])
delta50 = after50 - before50
direction = 'higher' if delta50 > 0 else 'lower' if delta50 < 0 else 'similar'
print(f'Observed Precision@50: before={before50:.2f}%, after={after50:.2f}%, difference={delta50:+.2f} percentage points.')
print(f'Safe claim: On this tested March→April population, measured Precision@50 was {direction} under client-grouped validation than under the original random row split. This is directional evidence for decision-support, not a guarantee of future page-level outcomes or business impact.')

Observed Precision@50: before=64.00%, after=64.00%, difference=+0.00 percentage points.
Safe claim: On this tested March→April population, measured Precision@50 was similar under client-grouped validation than under the original random row split. This is directional evidence for decision-support, not a guarantee of future page-level outcomes or business impact.


## Self-check

- [x] Two specific research findings are named and each has a concrete methodology question
- [x] Runtime → Run all completes after adding `HF_TOKEN` to Colab Secrets
- [x] Before/after comparison uses the established March→April lane and Week-5 Logistic Regression
- [x] Honest validation is grouped by `client_hash_id`; every fold has zero client overlap
- [x] April/future/label-derived fields are excluded from predictive features
- [x] Client/content IDs are used only for grouping/audit
- [x] Real failure examples contain no client names, URLs, or private queries
- [x] Claims use observed/measured/directional/decision-support language
- [x] The executed notebook is committed under `work/notebooks/w06_validation_audit.ipynb`